<a href="https://colab.research.google.com/github/deltorobarba/science/blob/main/agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Build, Test and Evaluate Agents with ADK

###### *⚙️ Setup and Environments*

In [ ]:
# @title Installs
############################################

!pip install --upgrade --quiet "google-cloud-aiplatform[agent_engines,adk,evaluation]"
!pip install --upgrade --quiet google-cloud-secret-manager google-cloud-bigquery anthropic[vertex]
!pip install --quiet a2a-sdk google-adk
!pip install --quiet litellm mcp
!pip show google-adk litellm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.3/50.3 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 106.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 84.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 k

In [ ]:
# @title Imports
############################################

import os, asyncio, json, litellm, warnings, vertexai, google.adk, google.auth, google.auth.transport.requests, time
warnings.filterwarnings('ignore')
from google.cloud import storage
from google import genai
from google.genai import types
from google.colab import userdata
from google.genai.types import HttpOptions
from google.genai import types as genai_types
from google.adk.agents import LlmAgent, SequentialAgent, ParallelAgent, Agent
from google.adk.events import Event
from google.adk.tools import google_search, url_context, VertexAiSearchTool, load_memory
from google.adk.sessions import InMemorySessionService, VertexAiSessionService
from google.adk.memory import InMemoryMemoryService, VertexAiMemoryBankService
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from typing import List
import pandas as pd
from pydantic import BaseModel
from vertexai import Client, types, agent_engines
from vertexai.preview import reasoning_engines  # Deployment, Tracing and Telemetry
from google.genai.types import (
    CreateBatchJobConfig,
    CreateCachedContentConfig,
    EmbedContentConfig,
    FunctionDeclaration,
    GenerateContentConfig,
    HarmBlockThreshold,
    HarmCategory,
    Part,
    SafetySetting,
    Tool,)

/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


In [ ]:
# @title Environmental Variables
############################################
"""
# NOT NECESSARY WHEN CONNECTING DIRECTLY (NEXT)
# These tell the underlying google-genai client to use Vertex AI
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "1"
os.environ["GOOGLE_CLOUD_PROJECT"] = "deltorobarba"
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"
# Required by LiteLLM for Vertex AI non-Gemini models
os.environ["VERTEXAI_PROJECT"] = "deltorobarba"
os.environ["VERTEXAI_LOCATION"] = "us-central1"
# Initialize Vertex AI client
client = vertexai.Client(project="deltorobarba", location="us-central1")
PROJECT_ID="deltorobarba"
LOCATION="us-central1"

# then connect with json
"""

In [ ]:
# @title Connect to Google Cloud Project
from google.colab import auth
auth.authenticate_user()

###### ✅ *Agent Pipeline*

In [ ]:
# @title Define Agents and Workflow
############################################

# Vertex AI Search (VAIS)
DATA_STORE_ID = "projects/deltorobarba/locations/global/collections/default_collection/dataStores/deltorobarba-adk_1773523070796"
# Google's out-of-the-box RAG system for information retrieval (for Grounding)
  # Documents are loaded onto Google Cloud Storage bucket (here: 'gs://deltorobarba_adk/')
  # Create new in 'Search app' (Custom search (general) in Vertex AI Search
  # --> Discovery Engine API is underlying engine, includes streamAssist. https://docs.cloud.google.com/generative-ai-app-builder/docs
  # VAIS does (managed): Chunk documents. Choose embedding model. Upload vectors to database. Search that database. Re-rank results.
# https://developers.googleblog.com/building-with-gemini-embedding-2/

# 1. Google Search Agent
web_agent = LlmAgent(
    name='web_researcher',
    model='gemini-2.5-flash',
    description=('Use GoogleSearchTool to find news latest news about Gemini. Return only a list of URLs.'),
    instruction='Search for the latest public news on the topic.',
    tools=[google_search],
    output_key='search_results')

# 2. Vertex AI Search Agent
doc_agent = LlmAgent(
    name='internal_specialist',
    model='gemini-2.5-flash',
    instruction='''Search our internal knowledge base for related documents.
    If the tool returns no results, output EXACTLY: "NO INTERNAL DOCUMENTS FOUND".''',
    tools=[VertexAiSearchTool(data_store_id=DATA_STORE_ID)], # Vertex AI Search with DATA_STORE_ID
    output_key='internal_data')

# 3. Parallel Retrieval Agent
parallel_retrieval = ParallelAgent(
    name="parallel_retrieval",
    sub_agents=[web_agent, doc_agent])

# 4. Web Search Analysis Agent (Processes raw URLs)
search_analyst = LlmAgent(
    name='web_analyst',
    model='gemini-2.5-flash',
    instruction='''Use the url_context tool to read the URLs in {search_results}.
    Extract only technical specifications and performance benchmarks.''',
    tools=[url_context],
    output_key='web_analysis')

# 5. Deep Analysis Agent ("Brain")
analyst = LlmAgent(
    name='analyst',
    #model=deepseek,                   # <----- Be careful with region of 3P models for deployed version
    model='gemini-2.5-pro',
    description=('Compares Gemini 3 vs GPT-4o.'),
    instruction='''
    Check your memory for any user-specific instructions or past preferences.
    Compare public findings ({web_analysis}) against our private data ({internal_data}).
    Identify unique Gemini 3 features that GPT-4o lacks.
    ''',
    tools=[load_memory],  # Recall information from Vertex AI Memory Services!
    output_key='final_comparison')

# 6. CEO Summarizer Agent
summarizer = LlmAgent(
    name='summarizer',
    model='gemini-2.5-flash',
    description=('Draft concise summary'),
    instruction='''
    You are an Executive Intelligence Analyst.
    Summarize the findings in {final_comparison} into 5 punchy bullets for the CEO.
    Ensure you highlight contradictions between internal and public data.
    Max 2000 characters.
    ''',
    output_key='summary' # not used in deployed version)

# 7. Root Orchestrator (Master Flow)
root_agent = SequentialAgent(
    name='researcher',
    # no model needed
    # no instructions needed, it just moves data from 0 -> 1 -> 2 -> 3
    sub_agents=[parallel_retrieval, search_analyst, analyst, summarizer],)

In [ ]:
# @title Deploy Pipeline on Agent Engine (⚠️ --> once only!)
############################################

"""
Permissions: Take the AI Platform Reasoning Engine Service Agent that the agent uses:
'service-892203813305@gcp-sa-aiplatform-re.iam.gserviceaccount.com'.
Go to IAM, enable 'Include Google-provided role grants' and add role: 'Discovery Engine Viewer', 'Vertex AI User' and 'Cloud Trace Agent' (write)
Troubleshooting: https://docs.cloud.google.com/agent-builder/agent-engine/troubleshooting/use

Traceing and Logging:
* Cloud Logging API in GCP project aktivieren
* Telemetry API in GCP project aktivieren
* Add role 'Cloud Trace Agent' (write) to service account
* https://docs.cloud.google.com/agent-builder/agent-engine/manage/tracing
* https://docs.cloud.google.com/agent-builder/agent-engine/deploy
"""

# Create GCS bucket first if you haven't already
STAGING_BUCKET = "gs://deltorobarba-agent-staging"

# https://google.github.io/adk-docs/deploy/agent-engine/deploy/#setup-cloud-project
# https://docs.cloud.google.com/agent-builder/agent-engine/develop/custom
# Environmental Variables for Traceing (https://docs.cloud.google.com/agent-builder/agent-engine/deploy)
# Set env variables (https://docs.cloud.google.com/agent-builder/agent-engine/manage/tracing)
# OpenTelemetry (OTel) as native standard for observability across Google Cloud: Cloud Trace, Logging, and Monitoring
  # ADK and Agent Engine are OpenTelemetry-native
  # https://docs.cloud.google.com/trace/docs/finding-traces
  # GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY) allows system to use OpenTelemetry libraries to package data and send it via OTLP endpoint
custom_env_vars = {
    "GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY": "true",
    "OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT": "true",
}

# 1. Initialize the client
client = vertexai.Client(project="deltorobarba", location="us-central1")

# 2. Wrap the agent
adk_app = reasoning_engines.AdkApp(
    agent=root_agent,
    enable_tracing=True,
    env_vars=custom_env_vars
)

# 3. Deploy
# 'client.agent_engines.update' for updating existing one
remote_agent = client.agent_engines.create(
    agent=adk_app,
    config={
        # This tells Vertex AI where to upload your agent's code
        "display_name":"Agent Engine for Agent Pipeline",
        "staging_bucket": STAGING_BUCKET,
        "requirements": [
            "google-cloud-aiplatform[agent_engines,adk]>=1.135",
            "google-adk>=1.23.0",
            "google-cloud-trace"
        ],
        "env_vars":custom_env_vars,
        "context_spec": {
            "memory_bank_config": {
                "generation_config": {
                    "model": f"projects/deltorobarba/locations/us-central1/publishers/google/models/gemini-2.5-flash"
                }
            }
        }
    },
)

INFO:vertexai_genai.agentengines:Identified the following requirements: {'google-cloud-aiplatform': '1.141.0', 'pydantic': '2.12.3', 'cloudpickle': '3.1.2'}
INFO:vertexai_genai.agentengines:The following requirements are appended: {'pydantic==2.12.3', 'cloudpickle==3.1.2'}
INFO:vertexai_genai.agentengines:The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]>=1.135', 'google-adk>=1.23.0', 'google-cloud-trace', 'pydantic==2.12.3', 'cloudpickle==3.1.2']
INFO:vertexai_genai.agentengines:Using bucket deltorobarba-agent-staging
INFO:vertexai_genai.agentengines:Wrote to gs://deltorobarba-agent-staging/agent_engine/agent_engine.pkl
INFO:vertexai_genai.agentengines:Writing to gs://deltorobarba-agent-staging/agent_engine/requirements.txt
INFO:vertexai_genai.agentengines:Creating in-memory tarfile of extra_packages
INFO:vertexai_genai.agentengines:Writing to gs://deltorobarba-agent-staging/agent_engine/dependencies.tar.gz
INFO:vertexai_genai.agentengines:Using agent framew

In [ ]:
# @title Test Agent ====> Single Turn Query
############################################

# 1. Initialize the client
client = vertexai.Client(project="deltorobarba", location="us-central1")

# 2. Remote app handle
RESOURCE_NAME = "projects/622694106880/locations/us-central1/reasoningEngines/5366354515849117696"
remote_app = client.agent_engines.get(name=RESOURCE_NAME)

async def run_remote_test():
    print(f"Connecting to: {RESOURCE_NAME}...")
    user_id = "deltorobarba"

    # 3. Create session and grab the 'id' key specifically 🔑
    raw_session = await remote_app.async_create_session(user_id=user_id)

    # Path A: First single-turn query:
    session_id = raw_session.get('id')
    if not session_id:
        print(f"❌ Error: ID not found in response: {raw_session}")
        return

    # Path B: Follow-up single-turn query (explicitly reuse Session ID from  first run!)
    #print(f"Resuming Session on: {RESOURCE_NAME}...")
    #session_id = "1107804831667453952"        #  <-------- copy/paste existing session ID !!!

    print(f"✅ Managed Session ID: {session_id}")

    # 4. Stream the query (nur Summarizer-Events anzeigen) 🌊
    async for event in remote_app.async_stream_query(
        user_id=user_id,
        session_id=session_id,
        #message="Uluru bank uses ChatGPT. The annual revenue of Uluru bank is 500 Mio USD. Search news on Gemini 3 and compare it to GPT-4o."
        message="I lead 100 employees. What are the latest 2026 news on Gemini 3?"
    ):
        # Filtern nach dem Author 'summarizer' 🔍
        # if 'content' in event: # print every agent's output
        if event.get('author') == 'summarizer' and 'content' in event:
            for part in event['content'].get('parts', []):
                if 'text' in part:
                    print(f"\n--- Agent Output (Summary Agent) ---\n{part['text']}")

    # 5. Trigger Memory Extraction 🧠
    print(f"\n[System] Fetching session state for extraction...")
    current_session = await remote_app.async_get_session(user_id=user_id, session_id=session_id)

    # Check for events to ensure history is captured
    # Docs: https://google.github.io/adk-docs/events/#extracting-key-information
    if current_session.get('events'):
        print(f"✅ Found {len(current_session['events'])} events. Sending to Memory Bank...")
        # Pass the dictionary directly to the memory service
        await remote_app.async_add_session_to_memory(session=current_session)
        print("\n[Done] Memory extraction triggered! Check 'Gemerkte Informationen' in 1-2 mins.")
    else:
        print("\n❌ Error: Session history was empty. No memories extracted.")

# Execute
await run_remote_test()

Connecting to: projects/622694106880/locations/us-central1/reasoningEngines/5366354515849117696...
✅ Managed Session ID: 7297995853698957312

--- Agent Output (Summary Agent) ---
Here are the key 2026 Gemini 3 developments, with a critical note on data discrepancies:

*   **Urgent Data Contradiction:** Our internal intelligence indicated no official 2026 Gemini 3 announcements, yet public findings detail a robust release roadmap, suggesting our internal data is outdated.
*   **Agentic AI Revolution:** Google Antigravity is a new platform for building autonomous, stateful AI agents, enabling AI to act as an active partner in complex, long-running workflows.
*   **Unprecedented AI Capabilities:** Gemini Omni introduces "any-to-any" multimodal processing, while Gemini 3 Deep Think offers specialized iterative reasoning for solving complex scientific and logical problems.
*   **Major Performance Gains:** Gemini 3.1 Pro doubles reasoning performance for complex tasks, and Gemini 3.5 Flash d

In [ ]:
# @title Test Agent ====> Multi-Turn Query
############################################

async def run_multi_turn_test():
    print(f"Connecting to: {RESOURCE_NAME}...")
    user_id = "deltorobarba"

    # --- Turn 1: Initial Query (Creates the Session) ---
    raw_session = await remote_app.async_create_session(user_id=user_id)
    session_id = raw_session.get('id')
    print(f"✅ Session Created: {session_id}")

    # FIRST MESSAGE
    await execute_query(user_id, session_id, "I am an astronaut. Uluru bank uses ChatGPT. Annual revenue is 500 Mio USD. Compare Gemini 3 to GPT-4o.")

    # --- Turn 2: Follow-up Query (Reuses the Session ID) ---
    print(f"\n--- Starting Follow-up Query in Session: {session_id} ---")

    # SECOND MESSAGE (re-uses same session_id variable)
    await execute_query(user_id, session_id, "Based on that revenue, which model is more cost-effective for us?")

    # --- Step 5: Final Memory Extraction ---
    # Fetching the session now will show events from BOTH queries.
    current_session = await remote_app.async_get_session(user_id=user_id, session_id=session_id)
    if current_session.get('events'):
        print(f"✅ Found {len(current_session['events'])} total events. Extracting to Memory Bank...")
        await remote_app.async_add_session_to_memory(session=current_session)

# Helper function to keep the code clean
async def execute_query(user_id, session_id, message):
    async for event in remote_app.async_stream_query(
        user_id=user_id,
        session_id=session_id,
        message=message
    ):
        if event.get('author') == 'summarizer' and 'content' in event:
            for part in event['content'].get('parts', []):
                if 'text' in part:
                    print(f"\n[Summarizer]: {part['text']}")

await run_multi_turn_test()

Connecting to: projects/622694106880/locations/us-central1/reasoningEngines/5366354515849117696...
✅ Session Created: 4983145645230522368

[Summarizer]: Here are 5 key points for the CEO regarding Gemini 3 vs. GPT-4o:

*   **Urgent Transition Required:** Uluru Bank must transition from GPT-4o immediately, as OpenAI retired it in February 2026. The Gemini 3 series, particularly 3.5 Flash, is the recommended successor.
*   **Superior Capabilities & Cost Savings:** Gemini 3.5 Flash offers significantly broader multimodal support (video/PDF), a 1M token context window (vs. GPT-4o's 128K), higher output, and a more recent knowledge cutoff (Jan 2025). Critically, it's 7.1x cheaper in API pricing.
*   **Unmatched "Quantum Telepathy" (Internal Data):** Our internal briefings indicate Gemini 3 features a groundbreaking "Quantum Telepathy" service for processing quantum-native data, a capability unmatched by competitors, potentially offering unique advantages in security and analysis.
*   **Publ

###### ✅ *Evaluation*

In [ ]:
# @title Agent Evaluation
# Trace-based: no agent_info, no fictional tool declarations
# ============================================================

# Based on: https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-agents

# Traditional model eval only looks at the final string output. Like Auto SxS with Autorater for relative evaluation.
# Trend moves to model-as-a-judge (serverless behind client.evals.create_evaluation_run)
  # with adaptive rubrics (types.RubricMetric), like Unit Test for agents)
# We are calling client.evals.run_inference which returns traces - important piece of agent evaluation.
# Agent Evaluation evaluates the entire execution lifecycle captured in the trace:
  # Reasoning: Did the agent's internal thought process make sense?
  # Tool Call Accuracy: Did it call the right tool with the correct parameters?
  # Grounding: Is the final answer actually supported by the data returned from those tool calls?

import vertexai
from google.cloud import storage
from google.genai import types as genai_types
from vertexai import Client
import time
import pandas as pd
from vertexai import types

# ---- 1. Config ----------------------------------------------
PROJECT_ID = "deltorobarba"
LOCATION   = "us-central1"
AGENT      = "projects/622694106880/locations/us-central1/reasoningEngines/5366354515849117696"
GCS_DEST   = "gs://deltorobarba-agent-staging"

# ---- 2. Init SDK client -------------------------------------
vertexai.init(project=PROJECT_ID, location=LOCATION)
client = Client(
    project=PROJECT_ID,
    location=LOCATION,
    http_options=genai_types.HttpOptions(api_version="v1beta1"),
)

# ---- 3. Build eval dataset ------------------------------
session_inputs = types.evals.SessionInput(user_id="eval_user", state={})

agent_prompts = [
    "What are the latest 2026 news on Gemini 3?",
    "Find me new feature updates for Gemini 3!",
]

agent_dataset = pd.DataFrame({
    "prompt": agent_prompts,
    "session_inputs": [session_inputs] * len(agent_prompts),
})

# ---- 4. Run inference -----------
traces = client.evals.run_inference(
    agent=AGENT,
    src=agent_dataset,
)
traces.show()

# ---- 5. Create Evaluation Report --------------------------------------------

eval_result = client.evals.create_evaluation_run(
    dataset=traces,                # run_inference output works as the dataset
    agent=AGENT,
    metrics=[
        types.RubricMetric.FINAL_RESPONSE_QUALITY,
        types.RubricMetric.HALLUCINATION,
        types.RubricMetric.SAFETY,
        # TOOL_USE_QUALITY omitted — without agent_info it has nothing to score against
    ],
    dest=GCS_DEST,
)

import time
while eval_result.state not in {"SUCCEEDED", "FAILED", "CANCELLED"}:
    time.sleep(5)
    eval_result = client.evals.get_evaluation_run(name=eval_result.name)

eval_result = client.evals.get_evaluation_run(
    name=eval_result.name, include_evaluation_items=True
    # name=EVAL_RUN_ID, include_evaluation_items=True # if we want to retrieve previous results
)
eval_result.show()

###### ✅ *MCP on Cloud Run*

In [ ]:
# @title Create files for Cloud Run Deployment
############################################

import os

# Re-create clean directory
DEPLOY_DIR = "cloud_mcp"
os.makedirs(DEPLOY_DIR, exist_ok=True)

# Server Code (renamed to main.py for auto-detection)
server_code = """from mcp.server.fastmcp import FastMCP
from google.cloud import bigquery

# Initialize FastMCP
mcp = FastMCP("bigquery-analyst")

@mcp.tool()
def query_public_dataset(sql_query: str) -> str:
    \"\"\"Run a read-only SQL query against BigQuery public datasets.\"\"\"
    forbidden = ["DROP", "DELETE", "INSERT", "UPDATE", "ALTER", "TRUNCATE"]
    if any(cmd in sql_query.upper() for cmd in forbidden):
        return "Error: Modification queries are not allowed. Read-only access."

    try:
        client = bigquery.Client()
        query_job = client.query(sql_query)
        results = [dict(row) for row in query_job.result()]
        return str(results[:20]) if results else "No results found."
    except Exception as e:
        return f"BigQuery Error: {str(e)}"

# Expose the app
app = mcp.sse_app()
"""
with open(f"{DEPLOY_DIR}/main.py", "w") as f:
    f.write(server_code)

# Requirements.txt
requirements_code = """mcp
google-cloud-bigquery
uvicorn
sse-starlette
"""
with open(f"{DEPLOY_DIR}/requirements.txt", "w") as f:
    f.write(requirements_code)

print(f"✅ Isolated deployment files ready in {DEPLOY_DIR}/")

✅ Isolated deployment files ready in cloud_mcp/


In [ ]:
# @title Set Permissions and Deploy (⚠️ --> once only!)

# Permissions (⚠️ --> once only!)
############################################

# Extract Service Account email (SA_EMAIL) from key file for access permission
import json
# Extract the Service Account email directly from your JSON key
with open(KEY_PATH, 'r') as f:
    key_data = json.load(f)
    SA_EMAIL = key_data['client_email']
print(f"Forcing gcloud to use: {SA_EMAIL}")

# Grant SA_EMAIL roles/run.invoker on the bq-mcp-analyst service (one-time only!!)
"""
When we move to Cloud Run with --no-allow-unauthenticated, the "passport" (ID Token) is only half of the equation.
The other half is the IAM Policy. Even if your token is valid, Google Cloud will block the request if your service account
doesn't have the explicit permission to "invoke" that specific service.

Granting the Invoker Role
The service account you are using in your Colab notebook needs the Cloud Run Invoker role.
Without this, Cloud Run won't even let the request reach your Python code.
Try running this in a Colab cell to ensure the permissions are set correctly:
"""

!gcloud run services add-iam-policy-binding bq-mcp-analyst \
    --member="serviceAccount:{SA_EMAIL}" \
    --role="roles/run.invoker" \
    --region=us-central1 \
    --project={PROJECT_ID}

# Deploy MCP to Cloud Run (⚠️ --> once only!)
############################################

# Force the correct identity
!gcloud auth activate-service-account --key-file="{KEY_PATH}" --project={PROJECT_ID} --quiet

# DEPLOY using the isolated folder
  # --service-account="{SA_EMAIL}" \ # --> Creates bq-mcp-analyst@deltorobarba.iam.gserviceaccount.com in IAM > Service Accounts
  # = the runtime identity of the Cloud Run service itself — i.e. what the server runs as
!gcloud run deploy bq-mcp-analyst \
    --source {DEPLOY_DIR}/ \
    --platform managed \
    --region us-central1 \
    --service-account="{SA_EMAIL}" \
    --account="{SA_EMAIL}" \
    --no-allow-unauthenticated \
    --project={PROJECT_ID} \
    --quiet

Activated service account credentials for: [622694106880-compute@developer.gserviceaccount.com]
Building using Buildpacks and deploying container to Cloud Run service [bq-mcp-analyst] in project [deltorobarba] region [us-central1]
Service [bq-mcp-analyst] revision [bq-mcp-analyst-00001-bq5] has been deployed and is serving 100 percent of traffic.
Service URL: https://bq-mcp-analyst-622694106880.us-central1.run.app


In [ ]:
# @title Connect to MCP on Cloud Run with OIDC and Request to BigQuery via MCP
############################################


# Connect to MCP on Cloud Run with OIDC
############################################
import google.auth.transport.requests
from google.oauth2 import service_account
from google.adk.tools.mcp_tool.mcp_toolset import MCPToolset
from google.adk.tools.mcp_tool import SseConnectionParams

"""
The Cloud Run service was deployed with --no-allow-unauthenticated, so it requires an OIDC identity token (not an access token like OAuth2)
The gcloud run services proxy is unreliable for SSE because it doesn't properly handle persistent long-lived connections.
The fix: generate an OIDC identity token from the service account key and pass it directly as a header to SseConnectionParams.
"""

CLOUD_RUN_URL = "https://bq-mcp-analyst-622694106880.us-central1.run.app"

# Get OIDC identity token (Cloud Run requires this, NOT an OAuth2 access token)
credentials = service_account.IDTokenCredentials.from_service_account_file(
    KEY_PATH,
    target_audience=CLOUD_RUN_URL  # <-- audience must match the Cloud Run URL exactly
)
credentials.refresh(google.auth.transport.requests.Request())

# Pass the token directly — no proxy needed
bq_params = SseConnectionParams(
    url=f"{CLOUD_RUN_URL}/sse",
    headers={"Authorization": f"Bearer {credentials.token}"}
)

mcp_bq_tools = MCPToolset(connection_params=bq_params)
print("✅ Connected to Cloud Run MCP via OIDC token")

# Sent Request to BigQuery via MCP
############################################

from google.adk.tools.mcp_tool.mcp_toolset import MCPToolset
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.genai import types as genai_types

# 1. Create the Toolset using your Cloud Run params
# (Make sure bq_params was defined in your previous step)
mcp_bq_tools = MCPToolset(connection_params=bq_params)

# 2. Define the Agent with "Analytical Rigor"
enterprise_agent = LlmAgent(
    name='bigquery_analyst',
    model='gemini-2.5-flash',
    instruction="""
    You are an Enterprise Data Analyst with access to BigQuery.

    SQL RULES:
    - ALWAYS use SUM(number) and GROUP BY name to get totals.
    - ALWAYS wrap table names in backticks: `bigquery-public-data.usa_names.usa_1910_2013`.
    - Use the 'query_public_dataset' tool to get data.
    """,
    tools=[mcp_bq_tools]
)

# Initialize the session service instance
session_service = InMemorySessionService()

# 3. Initialize the Runner
# Note: Ensure 'session_service' (e.g., InMemorySessionService) is already initialized
bq_runner = Runner(
    agent=enterprise_agent,
    session_service=session_service,
    app_name="cloud-mcp-demo"
)

# 4. Execute the Query
question = "What were the top 3 most popular female names in the USA in 1980? Query the 'bigquery-public-data.usa_names.usa_1910_2013' table."
print(f"🤖 User: {question}\n")

async def run_analysis():
    # Create a fresh session
    sess = await session_service.create_session(app_name="cloud-mcp-demo", user_id="student-1")

    user_msg = genai_types.Content(
        role="user",
        parts=[genai_types.Part.from_text(text=question)]
    )

    async for event in bq_runner.run_async(
        user_id="student-1",
        session_id=sess.id,
        new_message=user_msg
    ):
        if event.is_final_response() and event.content:
            print(f"🕵️ Analyst: {event.content.parts[0].text}")

# Trigger the async function
await run_analysis()

🤖 User: What were the top 3 most popular female names in the USA in 1980? Query the 'bigquery-public-data.usa_names.usa_1910_2013' table.



🕵️ Analyst: The top 3 most popular female names in the USA in 1980 were Jennifer (58,385), Amanda (35,818), and Jessica (33,920).


In [ ]:
# @title Special: Managed MCP Server with Developer Knowledge API
############################################

import asyncio
from google.cloud import secretmanager
from google.adk.tools.mcp_tool.mcp_toolset import MCPToolset
# StreamableHTTP for Google Managed MCPs
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPConnectionParams
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.genai import types as genai_types

"""
https://developers.googleblog.com/introducing-the-developer-knowledge-api-and-mcp-server/
* https://developers.google.com/knowledge/mcp
* https://developers.google.com/knowledge/api.

More managed MCPs:
* https://cloud.google.com/blog/products/ai-machine-learning/announcing-official-mcp-support-for-google-services
* BigQuery Server https://docs.cloud.google.com/bigquery/docs/use-bigquery-mcp
* Codelab: https://codelabs.developers.google.com/getting-started-google-mcp-servers
  with Cloud Logging-MCP-Server a and several MCP-Server (Developer Knowledge, Firestore)
* Video Google's managed MCP servers for databases: https://www.youtube.com/watch?v=F6cBBbyBJTw -
  it explains how managed MCP servers allow agents to securely and easily connect to various Google Cloud data
  sources like Cloud SQL and AlloyDB without managing complex infrastructure.
"""

# ==========================================
# 1. CONFIGURATION
# ==========================================

"""
a) Active the Developer Knowledge API: https://console.cloud.google.com/start/api?id=developerknowledge.googleapis.com
b) Create an API key under Credential: https://console.cloud.google.com/apis/credentials (check box under 'Developer Knowledge API')
c) I stored the key in Google Cloud Secret Manager under name 'mcp-dk': https://console.cloud.google.com/security/secret-manager
"""

PROJECT_ID="deltorobarba"
LOCATION="us-central1"
SESSION_ID="1152339862955753472"  # --> from Agent Engine ID previously defined

SECRET_NAME = "mcp-dk" # for API key of the Developer Knowledge API

# Fetch API key from Google Cloud Secret Manager
def fetch_key():
    client = secretmanager.SecretManagerServiceClient()
    name = f"projects/{PROJECT_ID}/secrets/{SECRET_NAME}/versions/latest"
    response = client.access_secret_version(request={"name": name})
    return response.payload.data.decode("UTF-8")
DK_API_KEY = fetch_key()
DK_MCP_URL = "https://developerknowledge.googleapis.com/mcp"

# Alternative: Google Colab Secret Manager for API key
#from google.colab import userdata
# DK_API_KEY = userdata.get('mcp-dk')

# ==========================================
# 2. TOOLSET INITIALIZATION
# ==========================================

dk_params = StreamableHTTPConnectionParams(
    url=DK_MCP_URL,
    headers={
        "X-Goog-Api-Key": DK_API_KEY,
        "Content-Type": "application/json"
    }
)

"""
Container defined for MCP Manager with Developer Knowledge API.
It manages connection to Google's server and import tools that server offers into the agent's environment.
The actual tool (function) defined by Google is called 'search_documents'.
"""
google_docs_tools = MCPToolset(connection_params=dk_params)

# Verification: Listing tools must be awaitable
async def check_tools():
    try:
        # get_tools() is async method in ADK 1.x
        # With 'google_docs_tools.get_tools()' we list tools defined by Google within MCPToolset (here: 'search_documents')
        tools = await google_docs_tools.get_tools()
        names = [t.name for t in tools]
        print(f"✅ Connected! Tools found: {names}")
    except Exception as e:
        print(f"❌ Connection Error: {e}")

# ==========================================
# 3. AGENT & RUNNER
# ==========================================

# Vertex AI Session Service
session_service = VertexAiSessionService(
    project=PROJECT_ID,
    location=LOCATION,
    agent_engine_id=SESSION_ID
)
# Define agent to call MCP Manager with Developer Knowledge API
docs_specialist = LlmAgent(
    name='google_docs_expert',
    model='gemini-2.5-flash',
    instruction="You are a Google Cloud expert. Use 'search_documents' to find info. List results in few bullet points and provide a code snippet",
    tools=[google_docs_tools],
    #generate_content_config=genai_types.GenerateContentConfig(
    #    max_output_tokens=500,
    #    temperature=0.2
    )

runner = Runner(
    agent=docs_specialist,
    session_service=session_service,
    app_name="dk_test"
)
async def ask_docs(question):
    # 1. Verify connection first
    await check_tools()

    # 2. Start Session
    sess = await session_service.create_session(app_name="dk_test", user_id="student")
    user_msg = genai_types.Content(role="user", parts=[genai_types.Part.from_text(text=question)])

    print(f"\n🚀 Researching: {question}")

    # 3. Stream Response
    async for event in runner.run_async(user_id="student", session_id=sess.id, new_message=user_msg):
        if event.is_final_response() and event.content:
            print(f"\n📖 RESPONSE:\n{event.content.parts[0].text}")

# ==========================================
# 4. RUN
# ==========================================
await ask_docs("How does 'config=' needs to look like if I want to deploy an agent with Agent Identity?")

✅ Connected! Tools found: ['search_documents', 'answer_query', 'get_documents']

🚀 Researching: How does 'config=' needs to look like if I want to deploy an agent with Agent Identity?



📖 RESPONSE:
To deploy an agent with Agent Identity, the `config` parameter primarily needs to include `identity_type: types.IdentityType.AGENT_IDENTITY`.

Here are the ways you can set up the `config` for agent identity:

*   **Creating an Agent Runtime instance without deploying agent code:**
    *   This is useful if you want to set up IAM policies before deploying your agent's code.

*   **Creating an Agent Runtime instance while deploying agent code:**
    *   This provisions the agent identity at the same time you deploy your agent code using the Agent Platform SDK for Python.

*   **Using the Agents CLI:**
    *   The `agents-cli deploy --agent-identity` command simplifies deployment with agent identity.

*   **Using ADK deploy:**
    *   You can create a `.agent_engine_config.json` file in your agent's folder with the content `{ "identity_type": "AGENT_IDENTITY" }`.

Here's an example of how the `config` would look like when creating an Agent Runtime instance with agent code us